# Bangla RAG evaluation: Colab runner
Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Add your key under the key icon (Secrets) as `GEMINI_API_KEY` and switch on notebook access.

**Run order every session:** cell 1, cell 2, cell 2b (Drive), then the cell you need. Cell 2b makes `results/` and `human/` live on your Google Drive, so the Gemini cache and all outputs survive a disconnect and nothing is paid for twice. To switch Gemini accounts, just change the `GEMINI_API_KEY` secret; the cache is reused.

In [ ]:
# 1. Get the code (safe to re-run: clones once, then just pulls updates)
import os
REPO_URL = "https://github.com/mehediparvez/bangla_rag_evaluation.git"
if not os.path.isdir('/content/repo/.git'):
    !git clone $REPO_URL /content/repo
%cd /content/repo
!git pull -q
!pip -q install -r requirements.txt

In [ ]:
# 2. API key from Colab Secrets (never paste the key into a cell)
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
!python -m pytest -q

In [ ]:
# 2b. Persist results on Google Drive: results/ and human/ become links to Drive folders.
# Run this EVERY session before any run cell. Existing local files are moved to Drive first.
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
D = '/content/drive/MyDrive/bangla_rag_results'
for name in ['results', 'human']:
    target = f'{D}/{name}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(name):
        continue
    if os.path.isdir(name):
        shutil.copytree(name, target, dirs_exist_ok=True)
        shutil.rmtree(name)
    os.symlink(target, name)
print('results ->', os.path.realpath('results'))
print('cached Gemini answers:', sum(1 for _ in open('results/gemini_cache.jsonl', encoding='utf-8')) if os.path.exists('results/gemini_cache.jsonl') else 0)

In [ ]:
# 3. Download BanglaRQA
!python -m bangla_rag.data

In [ ]:
# 4. Retrieval-only metrics: free, no Gemini calls. Checks the dense models and FAISS work.
!python -m bangla_rag.run --preset core --retrieval-only --n 0 --out results

In [ ]:
# 5. Smoke test: 5 questions through Gemini, separate output folder
!python -m bangla_rag.run --runs none,hybrid-bge-m3_c512_k3 --n 5 --rpm 10 --out results_smoke
import json
for line in open('results_smoke/hybrid-bge-m3_c512_k3.jsonl', encoding='utf-8'):
    r = json.loads(line); print(r['question'], '->', r['answer'], '| refs:', r['references'], '| F1', round(r['f1'], 2))

In [ ]:
# 6. Full core run (resumable; if the daily free quota runs out, run this cell again later)
!python -m bangla_rag.run --preset core --n 200 --rpm 10 --out results
!python -m bangla_rag.analyze --results results

In [ ]:
# 7. Optional: the full ablation (adds embedders, chunk sizes, top-5)
!python -m bangla_rag.run --preset full --n 200 --rpm 10 --out results
!python -m bangla_rag.analyze --results results

In [ ]:
# 8. RQ2: create the human grading sheet, then download it and grade it by hand
!python -m bangla_rag.human sample --runs bm25_c512_k3,hybrid-bge-m3_c512_k3 --n 100
from google.colab import files
files.download('human/grading_sheet.csv')

In [ ]:
# 9. After grading: upload the filled sheet to human/grading_sheet.csv, then run RAGAS and compare
!pip -q install -r requirements-ragas.txt
!python -m bangla_rag.ragas_eval --tag en
!python -m bangla_rag.human analyze --ragas human/ragas_en.csv

In [ ]:
# 9b. RQ2 round 2: the first 100 grades were almost all 3s (ceiling effect), so add 100 NEW rows with
# half drawn from likely-bad answers (gold answer not in context, or F1 < 0.5). Old graded rows are kept.
# FIRST upload your GRADED first sheet to human/grading_sheet.csv (it must contain your grades).
!python -m bangla_rag.human sample --runs bm25_c512_k3,hybrid-bge-m3_c512_k3 --n 100 --hard-frac 0.5 --extend human/grading_sheet.csv --out human/grading_sheet_v2.csv
from google.colab import files
files.download('human/grading_sheet_v2.csv')
# Grade ONLY the new rows (row_id 101-200), save as CSV, upload it as human/grading_sheet_v2.csv, then run cell 9c.

In [ ]:
# 9c. Score the new rows (already-scored rows 1-100 are skipped) and compare with all 200 human grades
!python -m bangla_rag.ragas_eval --sheet human/grading_sheet_v2.csv --tag en --workers 1
!python -m bangla_rag.human analyze --sheet human/grading_sheet_v2.csv --ragas human/ragas_en.csv

In [ ]:
# 10. Nothing to copy: results/ and human/ are already stored on Drive (see cell 2b).
!ls -la /content/drive/MyDrive/bangla_rag_results/results